In [1]:
import pandas as pd
import numpy as np

# Set seed so data is consistent every time you run it
np.random.seed(42)

# Number of transactions
n = 1000

# Generate data
df = pd.DataFrame({
    'transaction_id': range(1, n+1),
    'date': pd.date_range(start='2025-01-01', periods=n, freq='6H'),
    'amount': np.round(np.random.exponential(scale=150, size=n), 2),
    'category': np.random.choice(['Travel', 'Retail', 'Food', 'Tech', 'Healthcare', 'Transport'], size=n, p=[0.28, 0.22, 0.18, 0.14, 0.10, 0.08]),
    'merchant': np.random.choice(['SkyBridge Travel', 'AirElite', 'StyleVault', 'TechNow', 'GourmetHub', 'MediPlus', 'RoadEasy', 'StayPremium'], size=n),
    'city': np.random.choice(['New York', 'Chicago', 'Miami', 'Los Angeles', 'Houston'], size=n),
    'status': np.random.choice(['approved', 'approved', 'approved', 'declined', 'flagged'], size=n)
})

print(df.shape)
df.head(10)

(1000, 7)


/tmp/ipykernel_11554/2454298138.py:13: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  'date': pd.date_range(start='2025-01-01', periods=n, freq='6H'),


,transaction_id,date,amount,category,merchant,city,status
0,1,2025-01-01 00:00:00,70.39,Travel,TechNow,Miami,approved
1,2,2025-01-01 06:00:00,451.52,Food,RoadEasy,Los Angeles,approved
2,3,2025-01-01 12:00:00,197.51,Healthcare,StyleVault,Houston,flagged
3,4,2025-01-01 18:00:00,136.94,Tech,RoadEasy,Miami,approved
4,5,2025-01-02 00:00:00,25.44,Tech,TechNow,New York,approved
5,6,2025-01-02 06:00:00,25.44,Food,MediPlus,Los Angeles,approved
6,7,2025-01-02 12:00:00,8.98,Tech,StayPremium,Houston,approved
7,8,2025-01-02 18:00:00,301.68,Healthcare,AirElite,New York,declined
8,9,2025-01-03 00:00:00,137.86,Travel,StayPremium,New York,approved
9,10,2025-01-03 06:00:00,184.69,Retail,StayPremium,New York,declined


In [2]:
print("Total rows:", len(df))
print("Total columns:", len(df.columns))
print("Column names:", list(df.columns))

Total rows: 1000
Total columns: 7
Column names: ['transaction_id', 'date', 'amount', 'category', 'merchant', 'city', 'status']


In [3]:
# Check for missing values
print("Missing values:")
print(df.isnull().sum())

# Check for duplicate transactions
print("\nDuplicate rows:", df.duplicated().sum())

# Check amount column - remove any negative or zero values
print("\nNegative/zero amounts:", (df['amount'] <= 0).sum())
df = df[df['amount'] > 0]

print("\nCleaned data shape:", df.shape)
df.head(5)

Missing values:
transaction_id    0
date              0
amount            0
category          0
merchant          0
city              0
status            0
dtype: int64

Duplicate rows: 0

Negative/zero amounts: 0

Cleaned data shape: (1000, 11)


,transaction_id,date,amount,category,merchant,city,status,month,month_num,hour,day_of_week
0,1,2025-01-01 00:00:00,70.39,Travel,TechNow,Miami,approved,January,1,0,Wednesday
1,2,2025-01-01 06:00:00,451.52,Food,RoadEasy,Los Angeles,approved,January,1,6,Wednesday
2,3,2025-01-01 12:00:00,197.51,Healthcare,StyleVault,Houston,flagged,January,1,12,Wednesday
3,4,2025-01-01 18:00:00,136.94,Tech,RoadEasy,Miami,approved,January,1,18,Wednesday
4,5,2025-01-02 00:00:00,25.44,Tech,TechNow,New York,approved,January,1,0,Thursday


In [4]:
# Convert date to proper datetime format
df['date'] = pd.to_datetime(df['date'])

# Extract useful time features
df['month'] = df['date'].dt.month_name()
df['month_num'] = df['date'].dt.month
df['hour'] = df['date'].dt.hour
df['day_of_week'] = df['date'].dt.day_name()

print("Total rows:", len(df))
print("Total columns:", len(df.columns))
print("Column names:", list(df.columns))

Total rows: 1000
Total columns: 11
Column names: ['transaction_id', 'date', 'amount', 'category', 'merchant', 'city', 'status', 'month', 'month_num', 'hour', 'day_of_week']


In [5]:
# Total spend by category
category_summary = df.groupby('category')['amount'].agg(
    total_spend='sum',
    avg_transaction='mean',
    transaction_count='count'
).round(2).reset_index()

# Sort by total spend
category_summary = category_summary.sort_values('total_spend', ascending=False)

print("Spend by category:")
print(category_summary)

Spend by category:
     category  total_spend  avg_transaction  transaction_count
5      Travel     39940.02           141.13                283
0        Food     28287.98           158.03                179
2      Retail     26634.84           132.51                201
3        Tech     25969.03           163.33                159
4   Transport     12551.74           141.03                 89
1  Healthcare     12492.12           140.36                 89


In [6]:
# Total spend by month
monthly_summary = df.groupby(['month_num', 'month'])['amount'].agg(
    total_spend='sum',
    transaction_count='count'
).round(2).reset_index()

# Sort by month number
monthly_summary = monthly_summary.sort_values('month_num')

print("Monthly spend trend:")
print(monthly_summary)

Monthly spend trend:
   month_num      month  total_spend  transaction_count
0          1    January     17352.10                124
1          2   February     15679.56                112
2          3      March     18374.54                124
3          4      April     20221.92                120
4          5        May     21066.98                124
5          6       June     15589.68                120
6          7       July     19014.80                124
7          8     August     15124.19                124
8          9  September      3451.96                 28


In [7]:
# Anomaly detection using standard deviation
# Any transaction more than 2 standard deviations above the mean is flagged

mean_amount = df['amount'].mean()
std_amount = df['amount'].std()

# Define anomaly threshold
threshold = mean_amount + (2 * std_amount)

print(f"Average transaction: ${mean_amount:.2f}")
print(f"Standard deviation: ${std_amount:.2f}")
print(f"Anomaly threshold: ${threshold:.2f}")

# Flag anomalies
df['is_anomaly'] = df['amount'] > threshold

# Show anomalies
anomalies = df[df['is_anomaly'] == True]
print(f"\nTotal anomalies detected: {len(anomalies)}")
print(anomalies[['transaction_id', 'date', 'amount', 'category', 'merchant', 'city']].head(10))

Average transaction: $145.88
Standard deviation: $145.88
Anomaly threshold: $437.63

Total anomalies detected: 53
     transaction_id                date  amount    category          merchant  \
1                 2 2025-01-01 06:00:00  451.52        Food          RoadEasy   
11               12 2025-01-03 18:00:00  525.53   Transport          RoadEasy   
33               34 2025-01-09 06:00:00  446.05      Travel          MediPlus   
34               35 2025-01-09 12:00:00  505.59      Travel          RoadEasy   
50               51 2025-01-13 12:00:00  523.92  Healthcare       StayPremium   
69               70 2025-01-18 06:00:00  650.12      Retail           TechNow   
139             140 2025-02-04 18:00:00  535.17        Food           TechNow   
140             141 2025-02-05 00:00:00  492.30  Healthcare          MediPlus   
154             155 2025-02-08 12:00:00  636.61        Food  SkyBridge Travel   
226             227 2025-02-26 12:00:00  541.85      Retail        GourmetHu

In [8]:
# Export main cleaned dataset
df.to_csv('transactions_clean.csv', index=False)

# Export category summary
category_summary.to_csv('category_summary.csv', index=False)

# Export monthly summary
monthly_summary.to_csv('monthly_summary.csv', index=False)

# Export anomalies only
anomalies.to_csv('anomalies.csv', index=False)

print("All files exported successfully!")
print(f"Main dataset: {len(df)} rows")
print(f"Anomalies: {len(anomalies)} rows")

All files exported successfully!
Main dataset: 1000 rows
Anomalies: 53 rows


In [11]:
from google.colab import files

files.download('transactions_clean.csv')
files.download('category_summary.csv')
files.download('monthly_summary.csv')
files.download('anomalies.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
# Fix month order
month_order = ['January', 'February', 'March', 'April', 'May', 'June',
               'July', 'August', 'September']

df['month'] = pd.Categorical(df['month'], categories=month_order, ordered=True)

# Re-export the clean file
df.to_csv('transactions_clean.csv', index=False)

from google.colab import files
files.download('transactions_clean.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
import os

files_list = ['transactions_clean.csv', 'category_summary.csv',
              'monthly_summary.csv', 'anomalies.csv']

for f in files_list:
    if os.path.exists(f):
        print(f"✓ {f} exists")
    else:
        print(f"✗ {f} NOT found")

✓ transactions_clean.csv exists
✓ category_summary.csv exists
✓ monthly_summary.csv exists
✓ anomalies.csv exists


In [15]:
from google.colab import files

files.download('category_summary.csv')
files.download('monthly_summary.csv')
files.download('anomalies.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>